# Finetuning a Small LLM for EMI Collection Conversations

This notebook demonstrates end-to-end finetuning of a small language model on cleaned EMI collection conversation data.

In [ ]:
# Install dependencies
%pip install transformers datasets peft accelerate bitsandbytes

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import random

In [ ]:
# Load cleaned data
conversations = []
with open('part_a/cleaned_conversations.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        conversations.append(json.loads(line))

print(f"Loaded {len(conversations)} conversations")

In [ ]:
# Load model and tokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Format data into chat format
def format_conversation(conv):
    messages = []
    for turn in conv['turns']:
        role = "user" if turn['role'] == 'customer' else "assistant"
        messages.append({"role": role, "content": turn['text']})
    return messages

# Create training examples (alternate turns as Q&A)
training_data = []
for conv in conversations[:50]:  # Use first 50 for training
    turns = conv['turns']
    for i in range(0, len(turns)-1, 2):
        user_msg = turns[i]['text'] if turns[i]['role'] == 'customer' else turns[i+1]['text']
        assistant_msg = turns[i+1]['text'] if turns[i]['role'] == 'customer' else turns[i]['text']
        training_data.append({
            "messages": [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": assistant_msg}
            ]
        })

print(f"Created {len(training_data)} training examples")

In [ ]:
# Tokenize data
def tokenize_function(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )

dataset = Dataset.from_list(training_data)
tokenized_dataset = dataset.map(tokenize_function, remove_columns=["messages"])
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=10,
    save_steps=50,
    evaluation_strategy="steps",
    save_total_limit=1,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

# Train
trainer.train()

In [ ]:
# Save LoRA adapter
model.save_pretrained("./lora_adapter")

In [ ]:
# Inference function
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

# Test prompts
test_prompts = [
    "EMI kab pay karoge?",
    "Sir, your EMI is due. Can you pay today?",
    "Bhai, payment kar do."
]

print("Before finetuning:")
for prompt in test_prompts:
    response = generate_response(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()

In [ ]:
# Load finetuned model for inference
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)
finetuned_model = PeftModel.from_pretrained(base_model, "./lora_adapter")

def generate_response_finetuned(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(finetuned_model.device)
    with torch.no_grad():
        outputs = finetuned_model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):].strip()

print("After finetuning:")
for prompt in test_prompts:
    response = generate_response_finetuned(prompt)
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print()